# 📤 Notebook 3 — Export CSV pour le Dashboard Full Stack

## 🎯 Objectif

Ce notebook prépare et exporte les fichiers CSV nécessaires au dashboard Streamlit du projet **Price Intelligence Platform**.

Les données proviennent du fichier :

```python
../scrapers/data/cleaned_prices.csv

In [3]:
print(df.columns)

Index(['name', 'price', 'category', 'source', 'url', 'scraped_at',
       'source_label', 'price_date'],
      dtype='object')


In [4]:
import pandas as pd
import numpy as np
import os

# Chargement CSV
CSV_PATH = '../scrapers/data/cleaned_prices.csv'

df = pd.read_csv(CSV_PATH)

print("Colonnes AVANT rename :")
print(df.columns)

# Rename des colonnes
df = df.rename(columns={
    'price': 'price_mad',
    'source': 'source_platform',
    'category': 'category_normalized'
})

print("\nColonnes APRÈS rename :")
print(df.columns)

# Conversion date
df['scraped_at'] = pd.to_datetime(df['scraped_at'])

# Variable temps
df['days_since_start'] = (
    df['scraped_at'] - df['scraped_at'].min()
).dt.days

print("\n✅ Données prêtes")

# Vérification finale
print(df[['price_mad', 'days_since_start']].head())

Colonnes AVANT rename :
Index(['name', 'price', 'category', 'source', 'url', 'scraped_at',
       'source_label'],
      dtype='object')

Colonnes APRÈS rename :
Index(['name', 'price_mad', 'category_normalized', 'source_platform', 'url',
       'scraped_at', 'source_label'],
      dtype='object')

✅ Données prêtes
   price_mad  days_since_start
0      949.0                 0
1      949.0                 0
2     1979.0                 0
3     1979.0                 0
4     2039.0                 0


In [1]:
import pandas as pd

df = pd.read_csv('../scrapers/data/cleaned_prices.csv')

print(df.columns)

df = df.rename(columns={
    'price': 'price_mad',
    'source': 'source_platform',
    'category': 'category_normalized'
})

print(df.columns)

df['scraped_at'] = pd.to_datetime(df['scraped_at'])

df['days_since_start'] = (
    df['scraped_at'] - df['scraped_at'].min()
).dt.days

print(df.head())

Index(['name', 'price', 'category', 'source', 'url', 'scraped_at',
       'source_label'],
      dtype='object')
Index(['name', 'price_mad', 'category_normalized', 'source_platform', 'url',
       'scraped_at', 'source_label'],
      dtype='object')
                                                name  price_mad  \
0              A07 – 6,7" – 64 GB + 4 GB Ram – Green      949.0   
1              A07 – 6,7" – 64 GB + 4 GB Ram – Green      949.0   
2  Galaxy A16 8GB + 256GB - Black - 2 ans de gara...     1979.0   
3  Galaxy A16 8GB + 256GB - Black - 2 ans de gara...     1979.0   
4               Redmi Note 15 6GB 128GB Glacier Blue     2039.0   

  category_normalized source_platform  \
0         smartphones           jumia   
1         smartphones           jumia   
2         smartphones           jumia   
3         smartphones           jumia   
4         smartphones           jumia   

                                                 url  \
0  https://www.jumia.ma/samsung-a07-67-64-gb

In [2]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

os.makedirs('../data', exist_ok=True)

# ── Chargement données ─────────────────────
CSV_PATH = '../scrapers/data/cleaned_prices.csv'

df = pd.read_csv(CSV_PATH)

print("✅ CSV chargé")
print(df.shape)

# Rename colonnes
df = df.rename(columns={
    'price': 'price_mad',
    'source': 'source_platform',
    'category': 'category_normalized'
})

# Conversion date
df['scraped_at'] = pd.to_datetime(df['scraped_at'])

# Date principale
df['price_date'] = df['scraped_at']

# Segments prix
df['price_segment'] = pd.cut(
    df['price_mad'],
    bins=[0, 200, 1000, 3000, 8000, 100000],
    labels=['budget', 'entry', 'mid_range', 'high_end', 'premium']
)

# Extraction marque automatique
if 'brand' not in df.columns:

    def extract_brand(name):
        name = str(name).lower()

        brands = {
            'Samsung': ['samsung'],
            'Apple': ['iphone', 'apple'],
            'Xiaomi': ['xiaomi', 'redmi'],
            'Oppo': ['oppo'],
            'Huawei': ['huawei'],
            'Tecno': ['tecno'],
            'Itel': ['itel']
        }

        for brand, keywords in brands.items():
            if any(k in name for k in keywords):
                return brand

        return 'Other'

    df['brand'] = df['name'].apply(extract_brand)

print("\n✅ Données prêtes")
print(df.head())

✅ CSV chargé
(2990, 7)

✅ Données prêtes
                                                name  price_mad  \
0              A07 – 6,7" – 64 GB + 4 GB Ram – Green      949.0   
1              A07 – 6,7" – 64 GB + 4 GB Ram – Green      949.0   
2  Galaxy A16 8GB + 256GB - Black - 2 ans de gara...     1979.0   
3  Galaxy A16 8GB + 256GB - Black - 2 ans de gara...     1979.0   
4               Redmi Note 15 6GB 128GB Glacier Blue     2039.0   

  category_normalized source_platform  \
0         smartphones           jumia   
1         smartphones           jumia   
2         smartphones           jumia   
3         smartphones           jumia   
4         smartphones           jumia   

                                                 url  \
0  https://www.jumia.ma/samsung-a07-67-64-gb-4-gb...   
1  https://www.jumia.ma/samsung-a07-67-64-gb-4-gb...   
2  https://www.jumia.ma/samsung-galaxy-a16-8gb-25...   
3  https://www.jumia.ma/samsung-galaxy-a16-8gb-25...   
4  https://www.jumia.ma/xiaom

In [3]:
# ============================================================
# EXPORT 1 — clean_summary_stats.csv
# ============================================================

summary = df.groupby(
    ['category_normalized', 'source_platform']
)['price_mad'].agg(
    product_count='count',
    mean_price='mean',
    median_price='median',
    std_price='std',
    min_price='min',
    max_price='max'
).reset_index()

summary.to_csv(
    '../scrapers/data/clean_summary_stats.csv',
    index=False
)

print("✅ clean_summary_stats.csv créé")
print(summary.head())

✅ clean_summary_stats.csv créé
  category_normalized source_platform  product_count    mean_price  \
0             laptops           jumia            477   3760.807128   
1             laptops         marjane             60   9348.500000   
2             laptops      micromagma            212  15240.320755   
3         smartphones           jumia            487    813.752115   
4         smartphones         marjane             60   1944.516667   

   median_price    std_price  min_price  max_price  
0        2183.0  6636.413712      179.0    52726.0  
1        8394.5  6588.019775       48.0    35630.0  
2       13127.0  9912.967340     1717.0    80776.0  
3         149.0  2179.023863       19.0    15975.0  
4        1299.0  2231.019479      141.0     9070.0  


In [4]:
# ============================================================
# EXPORT 2 — daily_prices_dashboard.csv
# ============================================================

daily = df.groupby(
    ['price_date', 'source_platform', 'category_normalized']
)['price_mad'].agg(
    avg_price='mean',
    median_price='median',
    min_price='min',
    max_price='max',
    obs_count='count'
).reset_index()

daily.to_csv(
    '../scrapers/data/daily_prices_dashboard.csv',
    index=False
)

print("✅ daily_prices_dashboard.csv créé")
print(daily.head())

✅ daily_prices_dashboard.csv créé
                        price_date source_platform category_normalized  \
0 2026-03-30 10:27:13.498929+00:00           jumia         smartphones   
1 2026-03-30 10:27:13.503471+00:00           jumia         smartphones   
2 2026-03-30 10:27:13.504469+00:00           jumia         smartphones   
3 2026-03-30 10:27:13.506469+00:00           jumia         smartphones   
4 2026-03-30 10:27:13.508468+00:00           jumia         smartphones   

   avg_price  median_price  min_price  max_price  obs_count  
0      949.0         949.0      949.0      949.0          2  
1      799.0         799.0      799.0      799.0          2  
2     1979.0        1979.0     1979.0     1979.0          2  
3     2039.0        2039.0     2039.0     2039.0          2  
4     1129.0        1129.0     1129.0     1129.0          2  


In [5]:
# ============================================================
# EXPORT 3 — price_alerts.csv
# ============================================================

product_daily = df.groupby(
    ['url', 'name', 'price_date']
)['price_mad'].mean().reset_index()

product_daily = product_daily.sort_values(
    ['url', 'price_date']
)

product_daily['prev_price'] = (
    product_daily.groupby('url')['price_mad']
    .shift(1)
)

product_daily['price_change_pct'] = (
    (
        product_daily['price_mad']
        - product_daily['prev_price']
    )
    /
    product_daily['prev_price']
) * 100

alerts = product_daily[
    product_daily['price_change_pct'].abs() > 10
]

alerts.to_csv(
    '../scrapers/data/price_alerts.csv',
    index=False
)

print("✅ price_alerts.csv créé")
print(alerts.head())

✅ price_alerts.csv créé
Empty DataFrame
Columns: [url, name, price_date, price_mad, prev_price, price_change_pct]
Index: []


In [6]:
# ============================================================
# EXPORT 4 — brand_stats.csv
# ============================================================

# Extraction simple de marque depuis le nom

def extract_brand(name):

    name = str(name).lower()

    if 'samsung' in name:
        return 'Samsung'

    elif 'apple' in name or 'iphone' in name:
        return 'Apple'

    elif 'xiaomi' in name:
        return 'Xiaomi'

    elif 'oppo' in name:
        return 'Oppo'

    else:
        return 'Other'


df['brand'] = df['name'].apply(extract_brand)

brand_stats = df.groupby(
    ['brand', 'category_normalized']
)['price_mad'].agg(
    mean_price='mean',
    median_price='median',
    product_count='count'
).reset_index()

brand_stats.to_csv(
    '../scrapers/data/brand_stats.csv',
    index=False
)

print("✅ brand_stats.csv créé")
print(brand_stats.head())

✅ brand_stats.csv créé
   brand category_normalized    mean_price  median_price  product_count
0  Apple             laptops  20246.864865       19536.0             74
1  Apple         smartphones   5689.849315        4645.0             73
2   Oppo         smartphones   3734.052632        3799.0             19
3  Other             laptops   6034.000000        3056.0            670
4  Other         smartphones   1574.499621         335.0            581


In [7]:
# ============================================================
# RÉCAP FINAL
# ============================================================

files = [
    'clean_summary_stats.csv',
    'daily_prices_dashboard.csv',
    'price_alerts.csv',
    'brand_stats.csv'
]

print("=== EXPORTS GÉNÉRÉS ===")

for f in files:

    path = f'../scrapers/data/{f}'

    if os.path.exists(path):
        print(f'✅ {f}')
    else:
        print(f'❌ {f}')

=== EXPORTS GÉNÉRÉS ===
✅ clean_summary_stats.csv
✅ daily_prices_dashboard.csv
✅ price_alerts.csv
✅ brand_stats.csv
